## ✅ Resumen

**Lo que hizo este notebook:**

1. ✅ Configuraste el ticker: `{TICKER_TO_PROCESS}`
2. ✅ Llamó a DataBursátil API `/v2/historicos`
3. ✅ Recibió {len(data) if 'data' in locals() and data else 0} registros
4. ✅ Guardó respuesta cruda en Bronze layer

**Próximos pasos:**

1. Ejecuta `02_normalize_to_silver.ipynb` para parsear el JSON
2. Ejecuta `03_feature_engineering.ipynb` para calcular indicadores

**Diferencia con versión multi-ticker:**
- ❌ Multi: Lee `silver.tickers`, itera todos, guarda N registros
- ✅ Single: Procesa 1 ticker específico, guarda 1 registro

In [ ]:
# Crear registro para bronze.api_responses
record = {
    "response_id": str(uuid.uuid4()),
    "endpoint": "/v2/historicos",
    "request_params": {
        "emisora_serie": TICKER_TO_PROCESS,
        "inicio": start_date.isoformat(),
        "final": end_date.isoformat()
    },
    "response_body": api_response_json,
    "response_status": response.status_code if 'response' in locals() else None,
    "response_headers": dict(response.headers) if 'response' in locals() else {},
    "ingestion_timestamp": datetime.now(timezone.utc),
    "data_date": end_date,
    "ticker": TICKER_TO_PROCESS,
    "is_successful": api_success,
    "error_message": api_error
}

print(f"\n📝 Registro creado:")
print(f"   response_id: {record['response_id']}")
print(f"   ticker: {record['ticker']}")
print(f"   is_successful: {record['is_successful']}")
print(f"   data_date: {record['data_date']}")

# EN DATABRICKS: Guardar en Delta Lake
# df = spark.createDataFrame([record])
# df.write.format("delta").mode("append").saveAsTable("bronze.api_responses")
# print("✅ Guardado en bronze.api_responses")

# LOCAL: Guardar como JSON
import os
os.makedirs("output", exist_ok=True)
with open(f"output/{TICKER_TO_PROCESS.replace('*', '')}_bronze.json", "w") as f:
    json.dump(record, f, indent=2, default=str)
print(f"✅ Guardado en output/{TICKER_TO_PROCESS.replace('*', '')}_bronze.json")

## 💾 Guardar en Bronze Layer

**Explicación:**
1. Creamos un registro con metadata
2. Lo guardamos en `bronze.api_responses` (Delta Lake)
3. Este registro queda como "audit trail" completo

In [ ]:
# Calcular fechas
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"📅 Rango: {start_date} → {end_date}\n")

# Construir URL de la API
base_url = "https://api.databursatil.com/v2/historicos"
params = {
    "token": TOKEN,
    "emisora_serie": TICKER_TO_PROCESS,
    "inicio": start_date.isoformat(),
    "final": end_date.isoformat()
}

# Hacer request
print(f"🌐 Llamando API para {TICKER_TO_PROCESS}...")
print(f"   URL: {base_url}")
print(f"   Params: emisora_serie={TICKER_TO_PROCESS}, inicio={start_date}, final={end_date}\n")

try:
    response = requests.get(base_url, params=params, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        print(f"✅ Éxito! Recibidos {len(data)} registros")
        print(f"   Tamaño respuesta: {len(response.text)} bytes\n")
        
        # Mostrar primeros 3 registros
        print("📊 Primeros registros:")
        for i, (fecha, valores) in enumerate(list(data.items())[:3]):
            print(f"   {fecha}: precio={valores.get('precio')}, importe={valores.get('importe')}")
        
        api_response_json = response.text
        api_success = True
        api_error = None
        
    else:
        print(f"❌ Error HTTP {response.status_code}")
        print(f"   Respuesta: {response.text[:200]}")
        api_response_json = response.text
        api_success = False
        api_error = f"HTTP {response.status_code}"
        
except Exception as e:
    print(f"❌ Excepción: {str(e)}")
    api_response_json = None
    api_success = False
    api_error = str(e)

## 🌐 Llamada a la API

**Explicación:**
1. Construimos la URL con parámetros
2. Hacemos GET request
3. Guardamos la respuesta JSON

In [ ]:
import requests
import json
import uuid
from datetime import datetime, timezone, timedelta

# Para Databricks (comentar si corres local)
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

print("✓ Librerías importadas")

## 📦 Setup: Librerías y Spark

In [ ]:
# ============================================
# CONFIGURACIÓN: Cambia esto para procesar otro ticker
# ============================================

TICKER_TO_PROCESS = "WALMEX*"  # ← CAMBIA ESTO
LOOKBACK_DAYS = 30             # Cuántos días atrás
TOKEN = "763d9cdce4895c32fcb2f63358c6de"  # ← TU TOKEN DE DATABURSÁTIL

print(f"🎯 Ticker a procesar: {TICKER_TO_PROCESS}")
print(f"📅 Días históricos: {LOOKBACK_DAYS}")
print(f"🔑 Token: {TOKEN[:10]}...")

## ⚙️ Configuración del Ticker

**AQUÍ defines qué ticker procesar**